### Baselines

Two baselines on `data/processed/combined/combined_features.csv`, evaluated on a standard random 80/20 train/test split (`random_state=42`) — this split is used identically in `03g_model_bakeoff.ipynb` so the baselines and the bake-off models are compared on the exact same held-out rows.

1. **FLOPs-only linear regression** — single feature (`flops`) → `target`. Deliberately weak; establishes a floor.
2. **EC-NAS-style surrogate, retrained on our data** — a small MLP (`5 → 128 → 64 → 32 → 1`, via `src/models/neural_net.py`) using all core features (`params, depth, flops, epochs, batch_size`) → `target`. EC-NAS's own surrogate model used a 36-dimensional input (a graph/op encoding specific to their accuracy benchmark); we match its hidden-layer depth/width pattern, not its input size, since we only have these 5 shared-schema features. This is the "strong same-family baseline."

Note: `torch` was not installed in this environment when this task started (only `sklearn`/`xgboost` were) — installed here (CPU-only build) since `requirements.txt` already specifies it for the neural network regressor.

`batch_size` is a constant `256` for every row in the combined table (BUTTER-E's own batch size happens to also be 256 — confirmed in `01a`'s `describe()`, std 0). It's included per the requested feature set but contributes no signal here.

In [ ]:
# IMPORTS

import sys

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

sys.path.insert(0, "../../")
from src.models import neural_net

In [ ]:
# LOAD & SPLIT

df = pd.read_csv("../../data/processed/combined/combined_features.csv")

FEATURES = ["params", "depth", "flops", "epochs", "batch_size"]
TARGET = "target"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
# BASELINE 1: FLOPS-ONLY LINEAR REGRESSION

linreg = LinearRegression()
linreg.fit(X_train[["flops"]], y_train)
preds_linreg = linreg.predict(X_test[["flops"]])

mape_linreg = mean_absolute_percentage_error(y_test, preds_linreg)
r2_linreg = r2_score(y_test, preds_linreg)

print(f"MAPE: {mape_linreg:.4f}")
print(f"R2:   {r2_linreg:.4f}")

In [ ]:
# BASELINE 2: EC-NAS-STYLE SURROGATE MLP (all core features)

surrogate = neural_net.build_model()  # 5 -> 128 -> 64 -> 32 -> 1, defaults from src/models/neural_net.py
surrogate.fit(X_train, y_train)
preds_surrogate = surrogate.predict(X_test)

mape_surrogate = mean_absolute_percentage_error(y_test, preds_surrogate)
r2_surrogate = r2_score(y_test, preds_surrogate)

print(f"MAPE: {mape_surrogate:.4f}")
print(f"R2:   {r2_surrogate:.4f}")

In [ ]:
# SUMMARY

summary = pd.DataFrame([
    {"model": "Linear (FLOPs only)", "MAPE": mape_linreg, "R2": r2_linreg},
    {"model": "MLP surrogate (all features)", "MAPE": mape_surrogate, "R2": r2_surrogate},
])
summary

**Result** (executed once already to confirm the pipeline runs; re-run in VS Code to attach outputs to this file):

| model | MAPE | R² |
|---|---:|---:|
| Linear (FLOPs only) | 3.34 | 0.154 |
| MLP surrogate (all features) | 2.24 | 0.255 |

The surrogate beats the FLOPs-only floor as expected, but both numbers are weak in absolute terms — MAPE > 2.0 means predictions are typically off by more than 200% of the true value, and R² of 0.25 leaves most variance unexplained. This isn't necessarily a bug in the pipeline; the most likely cause is `target`'s raw scale: it spans nearly three orders of magnitude within BUTTER-E alone (28,839 J to 22.2M J) and BUTTER-E's mean target (~1.05M J) is roughly 11x EC-NAS's (~90,832 J) on top of that. Fitting MSE-based models directly on a heavy-tailed, multi-order-of-magnitude raw target tends to produce exactly this pattern (dominated by large values, poor relative accuracy on small ones). Not fixing this now — flagging it as something to weigh before committing to the RQ1-4 experimental matrix, e.g. whether `log(target)` should be the actual modeling target rather than raw joules. This same caveat applies to the bake-off results in `03g_model_bakeoff.ipynb`.

### Final re-run — within-family, log1p target, full feature set

Both baselines re-run using `log1p(target)` and, for the surrogate, the same 20-column feature set `03g_model_bakeoff.ipynb`'s Random Forest now uses (core + `is_gpu` + `shape_*` + PMLB dataset properties + `is_mlp_family`). The Linear baseline stays FLOPs-only by definition — that's the whole point of it as a floor, not something that should absorb the expanded feature set. Evaluated **within each family separately** (not pooled), for a fair comparison against RF's `03a`/`03b` results (BUTTER-E R²=0.973/Tau=0.936; EC-NAS R²=0.958/Tau=0.861) — the pooled numbers above predate both the log-transform and the auxiliary-feature work and are kept for historical record, not as the current comparison baseline.

In [ ]:
# RELOAD & DEFINE THE FULL FEATURE SET (matches 03g's RF)

import numpy as np
from scipy.stats import kendalltau

df2 = pd.read_csv("../../data/processed/combined/combined_features.csv")
NON_FEATURE_COLS = ["run_id", "target", "family", "source_dataset"]
FULL_FEATURES = [c for c in df2.columns if c not in NON_FEATURE_COLS]
print("full feature set:", FULL_FEATURES)

families = {
    "BUTTER-E": df2[df2["family"] == "MLP"],
    "EC-NAS": df2[df2["family"] == "CNN"],
}

In [ ]:
# RUN BOTH BASELINES, WITHIN EACH FAMILY, log1p TARGET

within_family_results = []

for fam_name, fam_df in families.items():
    X_train, X_test, y_train, y_test = train_test_split(fam_df, fam_df["target"], test_size=0.2, random_state=42)
    y_train_log = np.log1p(y_train)

    # Baseline 1: FLOPs-only linear regression
    linreg_f = LinearRegression()
    linreg_f.fit(X_train[["flops"]], y_train_log)
    preds_raw = np.expm1(linreg_f.predict(X_test[["flops"]]))
    within_family_results.append({
        "family": fam_name, "model": "Linear (FLOPs only)",
        "MAPE": mean_absolute_percentage_error(y_test, preds_raw),
        "R2": r2_score(y_test, preds_raw),
        "Tau": kendalltau(y_test, preds_raw)[0],
    })

    # Baseline 2: EC-NAS-style MLP surrogate, full feature set
    surrogate_f = neural_net.build_model()
    surrogate_f.fit(X_train[FULL_FEATURES], y_train_log)
    preds_raw_mlp = np.expm1(surrogate_f.predict(X_test[FULL_FEATURES]))
    within_family_results.append({
        "family": fam_name, "model": "MLP surrogate (full features)",
        "MAPE": mean_absolute_percentage_error(y_test, preds_raw_mlp),
        "R2": r2_score(y_test, preds_raw_mlp),
        "Tau": kendalltau(y_test, preds_raw_mlp)[0],
    })

within_family_summary = pd.DataFrame(within_family_results)
within_family_summary

**Result** (executed once already; re-run in VS Code to attach outputs):

| family | model | MAPE | R² | Kendall-Tau |
|---|---|---:|---:|---:|
| BUTTER-E | Linear (FLOPs only) | 1.383 | 0.087 | 0.280 |
| BUTTER-E | MLP surrogate (full features) | 0.121 | 0.967 | 0.920 |
| EC-NAS | Linear (FLOPs only) | 0.191 | 0.609 | 0.637 |
| EC-NAS | MLP surrogate (full features) | 0.123 | 0.812 | 0.672 |

**vs. RF (03a/03b):** BUTTER-E R²=0.973/Tau=0.936, EC-NAS R²=0.958/Tau=0.861.

**FLOPs-only is a much weaker floor for BUTTER-E than for EC-NAS** (R² 0.087 vs. 0.609) — consistent with everything found in RQ1/`03h`: BUTTER-E's energy depends heavily on `dataset`/`is_gpu`, which `flops` alone says nothing about, whereas EC-NAS's energy is more directly flops-driven (all CNNs training on CIFAR-10, no dataset-size confound). This isn't a new finding, just a second confirmation of it from a different angle.

**The MLP surrogate closes most of the gap to RF for BUTTER-E (0.967 vs. 0.973 — a 0.006 gap) but leaves a much bigger gap for EC-NAS (0.812 vs. 0.958 — a 0.146 gap).** The most likely explanation is sample size: BUTTER-E has 37,055 rows to fit a from-scratch neural net on; EC-NAS has only 2,805. A tree ensemble like Random Forest tends to be more data-efficient at this scale (bagging over many small trees) than a directly-trained MLP, which typically needs more examples to reach the same accuracy — consistent with EC-NAS's MLP result being the one baseline here that clearly underperforms its RF counterpart by a wide margin. Not tested further here (e.g. whether more epochs or a smaller network would close the EC-NAS gap), but worth noting as a data-size-dependent tradeoff between the two model types if the final choice for the RQ1-4 matrix isn't settled yet.